In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [4]:
%%writefile pages/watch.py
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st

from src import model, ui
from src.data import load, load_pa
from src.splits import woba_weight as RESULT_WEIGHT

ui.setup('球季觀察室')
teams, batters, pitchers, const, lg=load()

team=st.pills('選擇球隊',teams['team'],default=teams['team'].iloc[0])
if team is None:
    st.stop()
T = teams.set_index("team").loc[team]
abbr, color = T["abbr"], T["color"]

Overwriting pages/watch.py


In [15]:
print(teams)
print(batters.columns.to_numpy)
print(pitchers.columns.to_numpy)
print(T)

   rank         team abbr   W   L  T    PCT    GB  DIFF team_id    color   RS  \
0     1          味全龍    龍  67  53  0  0.558   0.0    70   0MJKt  #bd2e34  441   
1     2  統一7-ELEVEn獅    獅  64  55  1  0.538   2.5    48   hThSB  #dd732c  450   
2     3         富邦悍將    悍  59  61  0  0.492   8.0   -24   TPKQm  #204e93  434   
3     4         樂天桃猿    猿  57  61  2  0.483   9.0   -53   1zODE  #671a19  390   
4     5         中信兄弟    象  56  62  2  0.475  10.0     9   HCHks  #f4cf46  509   
5     6         台鋼雄鷹    鷹  54  65  1  0.454  12.5   -50   LH4lt  #1e4639  456   

    RA    G    pythag      luck  
0  371  120  0.578423 -0.020423  
1  402  120  0.551421 -0.013421  
2  458  120  0.475395  0.016605  
3  443  120  0.441967  0.041033  
4  500  120  0.508161 -0.033161  
5  506  120  0.452543  0.001457  
<bound method IndexOpsMixin.to_numpy of Index(['AB', 'AVG', 'BB', 'CS', '2B', 'GIDP', 'H', 'HBP', 'HR', 'IBB', 'PA',
       'PH', 'R', 'RBI', 'SB', 'SF', 'SH', 'SO', '3B', 'games', 'name', 'abbr

In [14]:
%%writefile -a pages/watch.py
def team_table():
    rows=[]
    tp=pd.DataFrame(model.team_pitching(pitchers)).set_index('abbr')
    for _, t in teams.iterrows():
        b=batters[batters['abbr']==t['abbr']]
        sp=pitchers[(pitchers['abbr']==t['abbr']) & (pitchers['role']=='先發')]
        w=b['PA']
        rows.append({
            'team':t['team'],
            "打擊":(b['wOBA_calc']*w).sum()/ w.sum(),
            "長打":(b['ISO_calc']*w).sum()/w.sum(),
            '選球':((b['BBp']-b['Kp'])*w).sum()/w.sum(),
            "先發": -(sp["FIP_calc"] * sp["IP"]).sum() / sp["IP"].sum(),   # 負號：越低越好
            "牛棚": -tp.loc[t["abbr"], "bullpen_fip"],
            "守備": tp.loc[t["abbr"], "DER"]
        })
    df=pd.DataFrame(rows).set_index('team')
    return (50 + 15*(df-df.mean())/df.std().clip(10,90))


Appending to pages/watch.py


In [17]:
%%writefile -a pages/watch.py
scores = team_table()
c1, c2 = st.columns(2)
with c1:
    cats = list(scores.columns)
    fig = go.Figure()
    fig.add_scatterpolar(r=[50] * 7, theta=cats + cats[:1], name="聯盟平均",
                         line=dict(color="#8C9BB0", dash="dot"))
    vals = scores.loc[team].tolist()
    fig.add_scatterpolar(r=vals + vals[:1], theta=cats + cats[:1], fill="toself", name=team,
                         line=dict(color=color))
    fig.update_layout(polar=dict(bgcolor="rgba(0,0,0,0)",
                                 radialaxis=dict(range=[0, 100], showticklabels=False, gridcolor="#26384F"),
                                 angularaxis=dict(gridcolor="#26384F", linecolor="#26384F")),
                      title="六項戰力（聯盟平均 = 50）", height=380, paper_bgcolor="rgba(0,0,0,0)")
    st.plotly_chart(fig, width="stretch")
with c2:
    luck=T['luck']
    ui.kpi('實際勝率',ui.f3(T['PCT']))
    ui.kpi('畢氏勝率',ui.f3(T['pythag']))
    ui.kpi('實際 − 畢氏',ui.f3(luck),note=ui.chip("贏得比實力多" if luck > 0 else "贏得比實力少","hot" if luck > 0 else "cold"))

Appending to pages/watch.py
